In [ ]:
import sys
from contextlib import redirect_stdout
from pathlib import Path
import io

sys.path.insert(0, str(Path('../..').resolve()))

In [ ]:
import pandas as pd

df = pd.read_parquet('../ssi_all_features.parquet')
df.shape

In [ ]:
tmp = pd.read_csv('../../../../analytics_dbt/seeds/icd_10_to_cmqcc.csv')
diabetes_dx_codes = tmp.loc[tmp['category'].str.contains('diabetes'), 'dx'].tolist()
diabetes_dx_codes = [code.replace('.', '_') for code in diabetes_dx_codes]
diabetes_dx_codes

In [ ]:
dx_features = df.loc[df['description'] == 'diagnosis', 'feature_name'].unique().tolist()
glucose_features = df.loc[df['description'] == 'glucose', 'feature_name'].unique().tolist()
diabetes_dx_features = [x for x in dx_features if x.endswith('_has_dx') and (any(x.startswith(code) for code in diabetes_dx_codes))]
print(len(glucose_features), len(diabetes_dx_features))
diabetes_dx_features

In [ ]:
df_pivoted = df.pivot(index='birthid', columns='feature_name', values='feature_value').reset_index()
df_pivoted.columns.name = ''
df_pivoted.shape

In [ ]:
from ssi.preprocess import preprocess

buf = io.StringIO()
with redirect_stdout(buf):
    df_pivoted_preprocessed = preprocess(df_pivoted, defer_imputation=True)

In [ ]:
df_pivoted_preprocessed.isna().sum().sort_values(ascending=False)

In [ ]:
from ssi.feature_filter import feature_filter

excluded = ['target_variable', 'birthid', 'mother_patid']
X = df_pivoted_preprocessed.drop(columns=excluded)
y = df_pivoted_preprocessed['target_variable']
mother_patid = df_pivoted_preprocessed['mother_patid']

# Use the filtered schema to define predictors. Imputation runs inside CV.
with redirect_stdout(io.StringIO()):
    df_pivoted_filtered = feature_filter(X.copy(), skip_medadmin_ingredient=True)

In [ ]:
df_pivoted_filtered.isna().sum().sort_values(ascending=False)

In [ ]:
glucose_features = [x for x in glucose_features if x in df_pivoted_filtered.columns]
# Remove measurement availability along with the glucose values.
glucose_features += [f'{x}_isna' for x in glucose_features if f'{x}_isna' in df_pivoted_filtered]
diabetes_dx_features = [x for x in diabetes_dx_features if x in df_pivoted_filtered.columns]
len(glucose_features), len(diabetes_dx_features)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

from src.clinical_imputer import ClinicalImputer
from src.model_training import nested_cv_evaluation_feature_selection

param_grid = {'clf__min_samples_split': [20, 50, 70, 100, 200, 500]}
cv_settings = dict(groups=mother_patid, outer_splits=5, inner_splits=5, random_state=range(10))

def rf_pipeline(columns):
    return Pipeline([
        ('clinical', ClinicalImputer(columns=columns)),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')),
    ])

### 1 baseline model, just everything

In [ ]:
pcornet_features = df_pivoted_filtered.columns.tolist()
all_features = []
for x in pcornet_features:
    if 'medadmin_ingredient' in x:
        if 'has_medadmin_ingredient' in x or 'earliest_minute' in x:
            all_features.append(x)
    else:
        all_features.append(x)

print(len(all_features))

In [ ]:
model_features = all_features
print(f'Predictors: {len(model_features)}')
metrics_baseline = nested_cv_evaluation_feature_selection(
    X, y, rf_pipeline(model_features), param_grid, **cv_settings,
)
metrics_baseline['number_of_features'] = len(model_features)

In [ ]:
metrics_baseline['roc_auc'].mean()

In [ ]:
import shap

def get_shap(clf, X, max_display=30):
    explainer = shap.TreeExplainer(clf, data=X,
                                   model_output='probability', feature_perturbation='interventional')
    shap_values = explainer(X, check_additivity=True)
    shap_values.values = shap_values.values[:,:,1]
    shap.plots.beeswarm(shap_values, max_display=max_display)
    return shap_values

In [ ]:
clf = rf_pipeline(model_features).set_params(clf__min_samples_split=100)
clf.fit(X, y)
shap_values = get_shap(clf.named_steps['clf'], clf[:-1].transform(X), max_display=50)

### 2. remove diabetes diagnosis features

In [ ]:
model_features = [col for col in all_features if col not in diabetes_dx_features]
print(f'Predictors: {len(model_features)}')
metrics_2 = nested_cv_evaluation_feature_selection(
    X, y, rf_pipeline(model_features), param_grid, **cv_settings,
)
metrics_2['number_of_features'] = len(model_features)

In [ ]:
clf = rf_pipeline(model_features).set_params(clf__min_samples_split=100)
clf.fit(X, y)
shap_values = get_shap(clf.named_steps['clf'], clf[:-1].transform(X), max_display=50)

### 3. remove glucose features

In [ ]:
model_features = [col for col in all_features if col not in glucose_features]
print(f'Predictors: {len(model_features)}')
metrics_3 = nested_cv_evaluation_feature_selection(
    X, y, rf_pipeline(model_features), param_grid, **cv_settings,
)
metrics_3['number_of_features'] = len(model_features)

In [ ]:
clf = rf_pipeline(model_features).set_params(clf__min_samples_split=100)
clf.fit(X, y)
shap_values = get_shap(clf.named_steps['clf'], clf[:-1].transform(X), max_display=50)

### 4 remove glucose and diabetes features

In [ ]:
model_features = [col for col in all_features if col not in diabetes_dx_features + glucose_features]
print(f'Predictors: {len(model_features)}')
metrics_4 = nested_cv_evaluation_feature_selection(
    X, y, rf_pipeline(model_features), param_grid, **cv_settings,
)
metrics_4['number_of_features'] = len(model_features)

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

metrics_baseline['test'] = 'all features'
metrics_2['test'] = 'all features - diabetes_dx_features'
metrics_3['test'] = 'all features - glucose_features'
metrics_4['test'] = 'all features - diabetes_dx_features - glucose_features'
all_metrics = pd.concat([metrics_baseline, metrics_2, metrics_3, metrics_4], axis=0)

sns.barplot(all_metrics, x='roc_auc', y='test', errorbar=None)
plt.xlim([0.65, 0.72])

In [ ]:
results_dir = Path('results/diabetes_revision')
results_dir.mkdir(parents=True, exist_ok=True)
all_metrics.to_csv(results_dir / 'outer_fold_metrics.csv', index=False)
all_metrics.groupby('test')[['roc_auc', 'ap', 'sens_80']].mean().to_csv(
    results_dir / 'performance_summary.csv',
)
plt.savefig(results_dir / 'figure_6.png', bbox_inches='tight', dpi=200)

import json
from importlib.metadata import version

run_config = {
    'repeat_seeds': list(cv_settings['random_state']),
    'outer_splits': cv_settings['outer_splits'],
    'inner_splits': cv_settings['inner_splits'],
    'rf_grid': param_grid,
    'versions': {p: version(p) for p in ('numpy', 'pandas', 'scikit-learn')},
}
(results_dir / 'run_config.json').write_text(json.dumps(run_config, indent=2) + '\n')

In [ ]:
all_metrics.groupby('test')['roc_auc'].mean()

In [ ]:
# Match the folds by repeat and fold number. These differences are descriptive.
matched = all_metrics.pivot(index=['repeat', 'outer_fold'], columns='test', values='roc_auc')
if matched.isna().any().any():
    raise ValueError('Missing matched outer-fold ROC-AUC results.')
deltas = matched.drop(columns='all features').subtract(matched['all features'], axis=0)
deltas.to_csv(results_dir / 'paired_roc_auc_differences.csv')
deltas.agg(['mean', 'std'])